In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Configuración de la Sesión (Nombres de variables ajustados para v3.0.1)
spark = SparkSession.builder \
    .appName("Analisis_Categorias")\
    .config("spark.mongodb.input.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping")\
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1")\
    .getOrCreate()

# 2. Carga de datos
df = spark.read.format("com.mongodb.spark.sql.DefaultSource").load()  #Abrir versión

df.printSchema()
df.show(5)




print("=== LIMPIEZA DE DATOS ===\n")

# Renombrar columnas para que sean más claras
df_clean = df.withColumnRenamed("identificador", "item") \
             .withColumnRenamed("valor", "precio")

# Ver cuántos registros tenemos
total_registros = df_clean.count()
print(f"📊 Total de registros: {total_registros}")

# Ver datos limpios
print("\n--- DATOS LIMPIOS ---")
df_clean.select("item", "precio", "grupo", "fecha_captura").show(5)


print("=== ANÁLISIS POR CATEGORÍA ===\n")

# Si no hay columna 'categoria', la creamos basada en los datos
if 'categoria' not in df_clean.columns:
    df_clean = df_clean.withColumn("categoria", F.lit("Libros"))

reporte = df_clean.groupBy("categoria").agg(
    F.count("item").alias("total_productos"),
    F.avg("precio").alias("precio_promedio"),
    F.min("precio").alias("precio_minimo"),
    F.max("precio").alias("precio_maximo"),
    F.round(F.avg("precio"), 2).alias("precio_promedio_redondeado")
)

reporte.show()




print("=== DETECTIVE DE PRECIOS ===\n")

# Encontrar el producto más caro
producto_mas_caro = df_clean.orderBy(F.desc("precio")).limit(1)

print("El producto más caro es:")
producto_mas_caro.select(
    F.col("item").alias("Producto"),
    F.col("precio").alias("Precio"),
    F.col("grupo").alias("Grupo"),
    F.col("fecha_captura").alias("Fecha de captura")
).show(truncate=False)




print("=== CONSULTAS ADICIONALES ===\n")

# 1. Productos con precio mayor a 50
print("1. Productos con precio > £50:")
productos_caros = df_clean.filter(F.col("precio") > 50)
productos_caros.select("item", "precio").show()

# 2. Precio promedio general
precio_promedio = df_clean.agg(F.avg("precio").alias("promedio_general"))
print(f"\n2. Precio promedio general: £{precio_promedio.collect()[0][0]:.2f}")

# 3. Producto más barato
producto_mas_barato = df_clean.orderBy(F.asc("precio")).limit(1)
print("\n3. Producto más barato:")
producto_mas_barato.select("item", "precio").show()



print("=== CORRIGIENDO TICKET DE SALIDA ===\n")

# Actualizar el nombre del grupo
df_corrected = df_clean.withColumn("grupo", F.lit("E-commerce"))

# Verificar el cambio
print("Grupos disponibles:")
df_corrected.select("grupo").distinct().show()

# Ticket de Salida corregido
ticket = df_corrected.filter(F.col("grupo") == "E-commerce") \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("precio"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print("\n--- TICKET DE SALIDA CORREGIDO ---")
ticket.show()



root
 |-- _id: struct (nullable = true)
 |    |-- oid: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- identificador: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+-------------------+----------------+--------------------+-----+
|                 _id|      fecha_captura|           grupo|       identificador|valor|
+--------------------+-------------------+----------------+--------------------+-----+
|{69d6613f025aa92f...|2026-04-08 14:07:55|Nombre del grupo|A Light in the Attic|51.77|
|{69d6613f025aa92f...|2026-04-08 14:07:55|Nombre del grupo|  Tipping the Velvet|53.74|
|{69d6613f025aa92f...|2026-04-08 14:07:55|Nombre del grupo|          Soumission| 50.1|
|{69d6613f025aa92f...|2026-04-08 14:07:55|Nombre del grupo|       Sharp Objects|47.82|
|{69d6613f025aa92f...|2026-04-08 14:07:55|Nombre del grupo|Sapiens: A Brief ...|54.23|
+--------------------+-------------------+----------------+